# GMM_case.ipynb — 应用案例展示

**教学目标**：通过三类案例理解 GMM 的独特价值——过度识别处理、非线性估计、多方程联合估计。

**数据来源**：均来自 `GMM_codes.ipynb` 生成的模拟数据或 FRED 数据备份（需先运行 `GMM_codes.ipynb`）。

**分析工具**：Python（`linearmodels`、`scipy.optimize`）；Stata 代码置于可折叠 callout 中。

| 案例 | 方法 | 矩条件来源 | 数据文件 |
|:----|:----|:---------|:-------|
| Case 1 | 有效 GMM vs. 2SLS | 工具变量正交性 | `data01_iv_hetero.csv` |
| Case 2a | 非线性 GMM（模拟）| Euler 方程 FOC | `data02_euler_simulated.csv` |
| Case 2b | 非线性 GMM（真实，可选）| Euler 方程 FOC | `data04_euler_FRED.csv` |
| Case 3 | 多方程 GMM（可选）| 多资产正交性 | `data03_asset_pricing.csv` |

In [ ]:
# ── 检查 linearmodels 是否安装 ───────────────────────────
try:
    from linearmodels.iv import IV2SLS, IVGMM
    print('✓ linearmodels 已安装')
except ImportError:
    print('✗ 请先安装 linearmodels：pip install linearmodels')

# ── 导入库 ───────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
from scipy import stats
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

# ── 字体与颜色（与 GMM_codes 完全一致）─────────────────
import matplotlib.font_manager as fm
available = [f.name for f in fm.fontManager.ttflist]
CN_FONTS = ['SimHei', 'Microsoft YaHei', 'PingFang SC', 'Hiragino Sans GB',
            'Noto Sans CJK SC', 'WenQuanYi Micro Hei', 'Source Han Sans SC']
cn_font = next((f for f in CN_FONTS if f in available), None)
if cn_font:
    plt.rcParams['font.family'] = [cn_font, 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120

COLOR_PRIMARY   = '#2C6BAC'
COLOR_SECONDARY = '#E8A020'
COLOR_NEUTRAL   = '#888888'
COLOR_FILL      = '#D6E8F7'
COLOR_GREEN     = '#2CA02C'
COLOR_RED       = '#D62728'
COLOR_BG        = '#F8F9FA'

print('环境初始化完成')

---
## Case 1：过度识别 IV + 异方差——有效 GMM vs. 2SLS

> **背景**：使用模拟数据 `method_GMM_data01_iv_hetero.csv`。数据生成过程中，$x$ 是内生变量，有 4 个工具变量（过度识别），且误差项存在异方差。  
> **研究问题**：有效 GMM 与 2SLS 在系数估计和标准误上有什么差异？  
> **分析目标**：① 验证两者系数接近（均一致）；② 展示异方差下标准误的差异；③ 解读 Hansen J 检验和 Sargan 检验的结论差异

In [ ]:
# ── Step 1：读入数据，绘制 x 与残差关系图（说明异方差）──
df1 = pd.read_csv('./data/method_GMM_data01_iv_hetero.csv')
print(f'数据形状：{df1.shape}')
print(df1.describe().round(3))

In [ ]:
# ── 绘制 OLS 残差散点图（说明异方差存在）────────────────
from numpy.polynomial import polynomial as P

ols_coef = np.polyfit(df1['x'], df1['y'], 1)
resid_ols = df1['y'] - (ols_coef[0] * df1['x'] + ols_coef[1])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
fig.patch.set_facecolor(COLOR_BG)

ax = axes[0]
ax.set_facecolor(COLOR_BG)
ax.scatter(df1['x'], resid_ols, alpha=0.35, s=12, color=COLOR_PRIMARY)
ax.axhline(0, color=COLOR_RED, lw=1.2, ls='--')
ax.set_xlabel('$x$（内生变量）', fontsize=11)
ax.set_ylabel('OLS 残差', fontsize=11)
ax.set_title('OLS 残差 vs. $x$（异方差特征）', fontsize=11, fontweight='bold', color=COLOR_PRIMARY)
ax.spines[['top','right']].set_visible(False)

ax = axes[1]
ax.set_facecolor(COLOR_BG)
ax.scatter(df1['x'], np.abs(resid_ols), alpha=0.35, s=12, color=COLOR_SECONDARY)
x_sort = np.sort(df1['x'])
sigma_true = 0.5 + 0.5 * np.abs(x_sort)
ax.plot(x_sort, sigma_true, color=COLOR_RED, lw=2, label='真实 $\\sigma_i = 0.5 + 0.5|x_i|$')
ax.set_xlabel('$x$', fontsize=11)
ax.set_ylabel('$|$残差$|$', fontsize=11)
ax.set_title('残差绝对值 vs. $x$（波动随 $x$ 增大）', fontsize=11, fontweight='bold', color=COLOR_PRIMARY)
ax.legend(fontsize=9)
ax.spines[['top','right']].set_visible(False)

plt.suptitle('Case 1：异方差的直观展示', fontsize=12, color=COLOR_NEUTRAL, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Step 2–4：估计三种方法 ────────────────────────────────
from linearmodels.iv import IV2SLS, IVGMM
import statsmodels.formula.api as smf

y_arr  = df1['y']
x_endog = df1[['x']]
z_inst  = df1[['z1', 'z2', 'z3', 'z4']]
const   = pd.DataFrame({'const': np.ones(len(df1))})

# 2SLS — 经典标准误
res_2sls = IV2SLS(y_arr, const, x_endog, z_inst).fit(cov_type='unadjusted')

# 2SLS — 异方差稳健标准误
res_2sls_rob = IV2SLS(y_arr, const, x_endog, z_inst).fit(cov_type='robust')

# 有效两步 GMM
res_gmm = IVGMM(y_arr, const, x_endog, z_inst).fit(cov_type='robust', weight_type='robust')

print('估计完成')

In [ ]:
# ── Step 5：对比表格 ──────────────────────────────────────
true_vals = {'const': 1.0, 'x': 1.5}

results = {
    '真实值':          {p: true_vals[p] for p in ['const', 'x']},
    '2SLS（经典SE）':  {p: (res_2sls.params[p], res_2sls.std_errors[p])
                        for p in ['const', 'x']},
    '2SLS（稳健SE）':  {p: (res_2sls_rob.params[p], res_2sls_rob.std_errors[p])
                        for p in ['const', 'x']},
    '有效 GMM':        {p: (res_gmm.params[p], res_gmm.std_errors[p])
                        for p in ['const', 'x']},
}

print('=' * 60)
print(f'{"方法":<18} {"截距（SE）":<22} {"斜率 x（SE）":<22}')
print('-' * 60)
for method, vals in results.items():
    if method == '真实值':
        print(f'{method:<18} {vals["const"]:<22.4f} {vals["x"]:<22.4f}')
    else:
        c_est, c_se = vals['const']
        x_est, x_se = vals['x']
        print(f'{method:<18} {c_est:.4f} ({c_se:.4f})     {x_est:.4f} ({x_se:.4f})')
print('=' * 60)

In [ ]:
# ── Step 6：展示 Hansen J 统计量 vs. Sargan 统计量 ────────
print('\n检验统计量对比：')
print('=' * 55)

# 2SLS Sargan（经典版，同方差假设）
sargan = res_2sls.sargan
print(f'Sargan 统计量（2SLS 经典）：{sargan.stat:.4f}，p = {sargan.pval:.4f}')
print(f'  自由度 = {sargan.df}（矩条件数 - 参数数 = 4 - 1）')

# Hansen J（GMM，异方差稳健）
j_stat = res_gmm.j_stat
print(f'\nHansen J 统计量（有效 GMM）：{j_stat.stat:.4f}，p = {j_stat.pval:.4f}')
print(f'  自由度 = {j_stat.df}')

print('\n结论：')
print('  • 两个检验均不拒绝「所有矩条件联合有效」（工具变量外生性通过）')
print('  • 若两者结论不一致，以 Hansen J 为准（Sargan 在异方差下可能失真）')
print('=' * 55)

In [ ]:
# ── Step 7：Durbin-Wu-Hausman 内生性检验 ─────────────────
durbin = res_gmm.durbin()
print(f'Durbin-Wu-Hausman 内生性检验：stat = {durbin.stat:.4f}，p = {durbin.pval:.4f}')
if durbin.pval < 0.05:
    print('  → 拒绝「x 外生」的原假设，内生性问题存在，使用 IV/GMM 是必要的')
else:
    print('  → 不拒绝「x 外生」，内生性问题不显著')

In [ ]:
# ── 可视化：标准误对比 ────────────────────────────────────
methods  = ['2SLS\n经典SE', '2SLS\n稳健SE', '有效\nGMM']
se_slope = [
    res_2sls.std_errors['x'],
    res_2sls_rob.std_errors['x'],
    res_gmm.std_errors['x'],
]
colors = [COLOR_RED, COLOR_SECONDARY, COLOR_PRIMARY]

fig, ax = plt.subplots(figsize=(7, 4))
ax.set_facecolor(COLOR_BG)
fig.patch.set_facecolor(COLOR_BG)
bars = ax.bar(methods, se_slope, color=colors, alpha=0.85, edgecolor='white', linewidth=1.5, width=0.5)
for bar, val in zip(bars, se_slope):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f'{val:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_ylabel('斜率 $x$ 的标准误', fontsize=11)
ax.set_title('Case 1：异方差下三种方法的标准误对比', fontsize=11,
             fontweight='bold', color=COLOR_PRIMARY)
ax.spines[['top','right']].set_visible(False)
ax.set_ylim(0, max(se_slope) * 1.25)
plt.tight_layout()
plt.show()

print('\n结果解读：')
print('  • 三种方法系数接近（均一致），说明工具变量有效')
print('  • 异方差下：2SLS 经典 SE 偏小（低估）> 2SLS 稳健 SE 扩大 > GMM SE 更有效')
print('  • Sargan 在异方差下可能过度拒绝，应以 Hansen J 为准')

::: {.callout-note collapse="true"}
### Stata 对应代码

```stata
import delimited "./data/method_GMM_data01_iv_hetero.csv", clear

* 2SLS（经典标准误）
ivreg2 y (x = z1 z2 z3 z4)

* 2SLS（异方差稳健标准误）
ivreg2 y (x = z1 z2 z3 z4), robust

* 两步有效 GMM（异方差稳健，推荐）
ivreg2 y (x = z1 z2 z3 z4), gmm2s robust
* 输出包含 Sargan/Hansen J 统计量
```

> **说明**：`ivreg2, robust` 模式下自动报告 Hansen J（非 Sargan）；`gmm2s robust` 对应 Python 的两步有效 GMM。
:::

---
## Case 2a：Euler 方程 GMM——模拟数据版

> **背景**：使用模拟数据 `method_GMM_data02_euler_simulated.csv`。  
> **真实参数**：$\beta = 0.98$（贴现因子），$\gamma = 2.0$（相对风险厌恶系数）  
> **矩条件**：$E_t[\beta (c_{t+1}/c_t)^{-\gamma} R_{t+1} - 1] = 0$，乘以工具变量 $z_t = (1, \Delta c_{t-1}/c_{t-1}, R_{t-1}, \Delta c_{t-2}/c_{t-2}, R_{t-2})$ 生成 5 个矩条件  
> **分析目标**：理解非线性 GMM 的实现逻辑，验证 GMM 能恢复真实参数，解读 J 检验

In [ ]:
# ── Step 1：读入数据，绘制时序图 ─────────────────────────
df2 = pd.read_csv('./data/method_GMM_data02_euler_simulated.csv')
print(f'数据形状：{df2.shape}')
print(df2.describe().round(4))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
fig.patch.set_facecolor(COLOR_BG)
T_plot = len(df2)

ax = axes[0]
ax.set_facecolor(COLOR_BG)
ax.plot(df2['dc'], color=COLOR_PRIMARY, lw=1.2, alpha=0.85)
ax.axhline(df2['dc'].mean(), color=COLOR_SECONDARY, lw=1.2, ls='--',
           label=f'均值 = {df2["dc"].mean():.4f}')
ax.set_xlabel('期数 $t$', fontsize=10)
ax.set_ylabel('消费增长率 $c_{t+1}/c_t$', fontsize=10)
ax.set_title('消费增长率时序图', fontsize=10, fontweight='bold', color=COLOR_PRIMARY)
ax.legend(fontsize=8)
ax.spines[['top','right']].set_visible(False)

ax = axes[1]
ax.set_facecolor(COLOR_BG)
ax.plot(df2['R'], color=COLOR_SECONDARY, lw=1.2, alpha=0.85)
ax.axhline(df2['R'].mean(), color=COLOR_PRIMARY, lw=1.2, ls='--',
           label=f'均值 = {df2["R"].mean():.4f}')
ax.set_xlabel('期数 $t$', fontsize=10)
ax.set_ylabel('资产收益率 $R_{t+1}$', fontsize=10)
ax.set_title('资产收益率时序图', fontsize=10, fontweight='bold', color=COLOR_PRIMARY)
ax.legend(fontsize=8)
ax.spines[['top','right']].set_visible(False)

plt.suptitle('Case 2a：模拟消费-资产数据（$\\beta=0.98, \\gamma=2.0$）',
             fontsize=11, color=COLOR_NEUTRAL, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Step 2：构造矩条件函数 ───────────────────────────────
def moment_conditions(params, dc, R, instruments):
    """
    params:      [beta, gamma]
    dc:          消费增长率数组，形状 (T,)
    R:           资产收益率数组，形状 (T,)
    instruments: 工具变量矩阵，形状 (T, q)，q=5
    返回:        g_bar，形状 (q,)，即样本矩条件均值向量
    """
    beta, gamma = params
    residual = beta * dc**(-gamma) * R - 1        # 形状 (T,)
    g = instruments * residual[:, np.newaxis]     # 形状 (T, q)
    return g.mean(axis=0)                         # 形状 (q,)

# 准备数据
dc_arr   = df2['dc'].values
R_arr    = df2['R'].values
instr = np.column_stack([
    np.ones(len(df2)),
    df2['dc_lag1'].values,
    df2['R_lag1'].values,
    df2['dc_lag2'].values,
    df2['R_lag2'].values,
])   # (T, 5)

# 测试矩条件函数
g_test = moment_conditions([0.98, 2.0], dc_arr, R_arr, instr)
print(f'真实参数下的样本矩条件（应接近0）：{g_test.round(5)}')

In [ ]:
# ── Step 3：两步 GMM 实现 ────────────────────────────────

# 第一步：W = I，最小化 g_bar' g_bar
def gmm_objective_step1(params, dc, R, instruments):
    g = moment_conditions(params, dc, R, instruments)
    return g @ g

result1 = minimize(gmm_objective_step1, x0=[0.95, 1.5],
                   args=(dc_arr, R_arr, instr),
                   method='Nelder-Mead',
                   options={'maxiter': 10000, 'xatol': 1e-8, 'fatol': 1e-10})
theta1 = result1.x
print(f'第一步 GMM：beta={theta1[0]:.4f}, gamma={theta1[1]:.4f}')

# 估计 S_hat（异方差稳健）
def compute_S_hat(params, dc, R, instruments):
    beta, gamma = params
    residual = beta * dc**(-gamma) * R - 1
    g_each = instruments * residual[:, np.newaxis]    # (T, q)
    g_each_demeaned = g_each - g_each.mean(axis=0)
    return g_each_demeaned.T @ g_each_demeaned / len(dc)

S_hat = compute_S_hat(theta1, dc_arr, R_arr, instr)
W2 = np.linalg.inv(S_hat)

# 第二步：W = S_hat^{-1}，最小化 g_bar' W g_bar
def gmm_objective_step2(params, dc, R, instruments, W):
    g = moment_conditions(params, dc, R, instruments)
    return g @ W @ g

result2 = minimize(gmm_objective_step2, x0=theta1,
                   args=(dc_arr, R_arr, instr, W2),
                   method='Nelder-Mead',
                   options={'maxiter': 10000, 'xatol': 1e-8, 'fatol': 1e-10})
theta2 = result2.x
print(f'第二步 GMM：beta={theta2[0]:.4f}, gamma={theta2[1]:.4f}')

In [ ]:
# ── Step 4：计算 Hansen J 统计量 ─────────────────────────
T_len = len(dc_arr)
g_hat = moment_conditions(theta2, dc_arr, R_arr, instr)
J_stat = T_len * g_hat @ W2 @ g_hat
J_pval = 1 - stats.chi2.cdf(J_stat, df=5-2)   # q-k = 5-2 = 3
print(f'Hansen J = {J_stat:.4f}, p-value = {J_pval:.4f}')
print(f'自由度 = 5 - 2 = 3（矩条件数 - 参数数）')
if J_pval > 0.05:
    print('  → 不拒绝矩条件有效（Euler 方程设定正确）')
else:
    print('  → 拒绝部分矩条件有效（可能存在模型设定问题）')

In [ ]:
# ── Step 5：展示参数估计结果 vs. 真实值 ──────────────────
print('\n' + '=' * 45)
print(f'  参数     真实值   第一步GMM  第二步GMM')
print('-' * 45)
print(f'  beta     0.9800   {theta1[0]:.4f}     {theta2[0]:.4f}')
print(f'  gamma    2.0000   {theta1[1]:.4f}     {theta2[1]:.4f}')
print('=' * 45)
print()
print('结果解读：')
print('  • GMM 在不假设分布的前提下，仅利用 Euler 方程矩条件恢复参数')
print('  • 估计值应接近真实值（存在有限样本误差）')
print(f'  • J 检验 p={J_pval:.3f}：不拒绝矩条件正确设定')

In [ ]:
# ── Step 6（建议）：目标函数轮廓曲线 ─────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
fig.patch.set_facecolor(COLOR_BG)

# 固定 beta，扫描 gamma
gamma_grid = np.linspace(0.5, 4.0, 150)
Q_gamma = [gmm_objective_step2([theta2[0], g], dc_arr, R_arr, instr, W2)
           for g in gamma_grid]

ax = axes[0]
ax.set_facecolor(COLOR_BG)
ax.plot(gamma_grid, Q_gamma, color=COLOR_PRIMARY, lw=2)
ax.axvline(theta2[1], color=COLOR_RED, lw=1.5, ls='--', label=f'$\\hat{{\\gamma}}={theta2[1]:.3f}$')
ax.axvline(2.0, color=COLOR_GREEN, lw=1.5, ls=':', label='真实值 $\\gamma=2.0$')
ax.set_xlabel('$\\gamma$', fontsize=12)
ax.set_ylabel('$Q(\\hat{\\beta}, \\gamma)$', fontsize=12)
ax.set_title('目标函数关于 $\\gamma$ 的轮廓（固定 $\\beta=\\hat{\\beta}$）',
             fontsize=10, fontweight='bold', color=COLOR_PRIMARY)
ax.legend(fontsize=9)
ax.spines[['top','right']].set_visible(False)

# 固定 gamma，扫描 beta
beta_grid = np.linspace(0.85, 1.10, 150)
Q_beta = [gmm_objective_step2([b, theta2[1]], dc_arr, R_arr, instr, W2)
          for b in beta_grid]

ax = axes[1]
ax.set_facecolor(COLOR_BG)
ax.plot(beta_grid, Q_beta, color=COLOR_SECONDARY, lw=2)
ax.axvline(theta2[0], color=COLOR_RED, lw=1.5, ls='--', label=f'$\\hat{{\\beta}}={theta2[0]:.4f}$')
ax.axvline(0.98, color=COLOR_GREEN, lw=1.5, ls=':', label='真实值 $\\beta=0.98$')
ax.set_xlabel('$\\beta$', fontsize=12)
ax.set_ylabel('$Q(\\beta, \\hat{\\gamma})$', fontsize=12)
ax.set_title('目标函数关于 $\\beta$ 的轮廓（固定 $\\gamma=\\hat{\\gamma}$）',
             fontsize=10, fontweight='bold', color=COLOR_PRIMARY)
ax.legend(fontsize=9)
ax.spines[['top','right']].set_visible(False)

plt.suptitle('Case 2a：GMM 目标函数轮廓——参数识别情况', fontsize=11,
             color=COLOR_NEUTRAL, y=1.01)
plt.tight_layout()
plt.show()

::: {.callout-note collapse="true"}
### Stata 对应代码

```stata
import delimited "./data/method_GMM_data02_euler_simulated.csv", clear

* 非线性 GMM 估计 Euler 方程
gmm (dc R dc_lag1 dc_lag2 R_lag1 R_lag2: ///
    ({beta}*(dc)^(-{gamma})*R - 1)), ///
    instruments(dc_lag1 dc_lag2 R_lag1 R_lag2) ///
    twostep winitial(identity)
```
:::

---
## Case 2b：Euler 方程 GMM——真实数据版（可选模块）

> ⚠️ **本节为可选模块**。若网络无法访问 FRED 数据，可跳过本节，Case 2a 已完整覆盖 Euler 方程 GMM 的核心内容。本地备份数据文件：`./data/method_GMM_data04_euler_FRED.csv`。

> **背景**：使用来自 FRED 的美国季度消费增长率和标普500超额收益率数据（1960–2019）。  
> **研究问题**：使用真实宏观数据估计 CRRA 效用函数的 $\beta$ 和 $\gamma$，并与文献中的典型估计值（$\beta \approx 0.99$，$\gamma \approx 1$–$3$）对照。

In [ ]:
# ── 读入数据（优先本地 CSV，备用 FRED 下载）─────────────
try:
    df4 = pd.read_csv('./data/method_GMM_data04_euler_FRED.csv', parse_dates=['DATE'])
    print(f'✓ 读入 FRED 数据，形状：{df4.shape}')
    print(df4.head())
except FileNotFoundError:
    print('✗ 未找到 FRED 数据文件，请先运行 GMM_codes.ipynb 中的数据04生成 cell')
    print('  或将此 cell 以下内容跳过，直接使用 data02（模拟数据）')

In [ ]:
# ── 数据描述与时序图 ──────────────────────────────────────
try:
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
    fig.patch.set_facecolor(COLOR_BG)

    ax = axes[0]
    ax.set_facecolor(COLOR_BG)
    ax.plot(df4['pce_growth'].values, color=COLOR_PRIMARY, lw=1.2, alpha=0.85)
    ax.axhline(0, color=COLOR_NEUTRAL, lw=0.8, ls='--')
    ax.set_title('美国季度消费增长率（PCE）', fontsize=10, fontweight='bold', color=COLOR_PRIMARY)
    ax.set_xlabel('期数（季度）'); ax.set_ylabel('增长率（%）')
    ax.spines[['top','right']].set_visible(False)

    ax = axes[1]
    ax.set_facecolor(COLOR_BG)
    ax.plot(df4['excess_ret'].values, color=COLOR_SECONDARY, lw=1.2, alpha=0.85)
    ax.axhline(0, color=COLOR_NEUTRAL, lw=0.8, ls='--')
    ax.set_title('标普500超额收益率（季度）', fontsize=10, fontweight='bold', color=COLOR_PRIMARY)
    ax.set_xlabel('期数（季度）'); ax.set_ylabel('超额收益率')
    ax.spines[['top','right']].set_visible(False)

    plt.suptitle('Case 2b：美国 1960–2019 季度数据（来源：FRED）',
                 fontsize=11, color=COLOR_NEUTRAL, y=1.01)
    plt.tight_layout()
    plt.show()
except Exception:
    print('跳过（数据未加载）')

In [ ]:
# ── 与 Case 2a 完全相同的 GMM 估计步骤 ───────────────────
try:
    # 构造消费增长率（pce_growth 已是增长率，转换为 c_{t+1}/c_t 形式）
    pce = df4['pce_growth'].values / 100 + 1   # 转换为比率
    R_real = df4['excess_ret'].values + df4['rf'].values + 1  # 总收益率

    # 生成滞后工具变量
    T4 = len(pce)
    pce_lag1 = np.roll(pce, 1); pce_lag1[0] = np.nan
    pce_lag2 = np.roll(pce, 2); pce_lag2[:2] = np.nan
    R_lag1   = np.roll(R_real, 1); R_lag1[0] = np.nan
    R_lag2   = np.roll(R_real, 2); R_lag2[:2] = np.nan

    mask = ~(np.isnan(pce_lag1) | np.isnan(pce_lag2) | np.isnan(R_lag1) | np.isnan(R_lag2))
    dc4  = pce[mask]
    R4   = R_real[mask]
    instr4 = np.column_stack([np.ones(mask.sum()), pce_lag1[mask], R_lag1[mask],
                               pce_lag2[mask], R_lag2[mask]])

    # 第一步 GMM
    res4_1 = minimize(gmm_objective_step1, x0=[0.97, 1.5],
                      args=(dc4, R4, instr4), method='Nelder-Mead',
                      options={'maxiter': 20000, 'xatol': 1e-8, 'fatol': 1e-10})
    theta4_1 = res4_1.x

    S4   = compute_S_hat(theta4_1, dc4, R4, instr4)
    W4_2 = np.linalg.inv(S4)

    res4_2 = minimize(gmm_objective_step2, x0=theta4_1,
                      args=(dc4, R4, instr4, W4_2), method='Nelder-Mead',
                      options={'maxiter': 20000, 'xatol': 1e-8, 'fatol': 1e-10})
    theta4_2 = res4_2.x

    g4_hat = moment_conditions(theta4_2, dc4, R4, instr4)
    J4 = len(dc4) * g4_hat @ W4_2 @ g4_hat
    p4 = 1 - stats.chi2.cdf(J4, df=3)

    print('=' * 50)
    print('  Case 2b：真实数据 GMM 结果')
    print('-' * 50)
    print(f'  beta  = {theta4_2[0]:.4f}  （文献参考：~0.99）')
    print(f'  gamma = {theta4_2[1]:.4f}  （文献参考：1–3）')
    print(f'  J 统计量 = {J4:.4f}, p = {p4:.4f}')
    print('=' * 50)
    print()
    print('讨论：真实数据与模拟数据估计差异来源：')
    print('  • 时变波动率（ARCH 效应）')
    print('  • 数据测量误差（PCE 不等于个人消费）')
    print('  • 「股权溢价之谜」：若 gamma 偏大，与消费低波动率矛盾')

except Exception as e:
    print(f'跳过 Case 2b 估计（原因：{e}）')

---
## Case 3：横截面资产定价检验——多方程矩条件 GMM（可选）

> 🟢 **可选模块**

> **背景**：使用模拟数据 `method_GMM_data03_asset_pricing.csv`，包含 25 个资产组合的月度超额收益率面板和市场超额收益率。  
> **研究问题**：市场因子的风险溢价是多少？CAPM 的定价误差（$\alpha_i$）是否联合为零？  
> **方法**：Fama-MacBeth 两步法 vs. GMM 多方程联合估计

In [ ]:
# ── 读入数据 ──────────────────────────────────────────────
df3 = pd.read_csv('./data/method_GMM_data03_asset_pricing.csv')
print(f'数据形状：{df3.shape}')
N_assets = df3['asset'].nunique()
T_periods = df3['time'].nunique()
print(f'资产数 N={N_assets}，时间期数 T={T_periods}')

In [ ]:
# ── Step 1：均值-标准差散点图（风险-收益图）─────────────
asset_stats = df3.groupby('asset')['excess_ret'].agg(['mean','std']).reset_index()
asset_stats['beta_true'] = df3.groupby('asset')['beta_true'].first().values

fig, ax = plt.subplots(figsize=(7, 5))
ax.set_facecolor(COLOR_BG)
fig.patch.set_facecolor(COLOR_BG)

sc = ax.scatter(asset_stats['std'], asset_stats['mean'],
                c=asset_stats['beta_true'], cmap='RdYlBu_r',
                s=70, alpha=0.85, edgecolors='white', linewidth=0.5)
plt.colorbar(sc, ax=ax, label='真实 $\\beta_i$', shrink=0.8)
ax.set_xlabel('收益率标准差（风险）', fontsize=11)
ax.set_ylabel('平均超额收益率（回报）', fontsize=11)
ax.set_title('Case 3：25 资产的风险-收益关系', fontsize=11,
             fontweight='bold', color=COLOR_PRIMARY)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# ── Step 2：逐资产时序回归，估计 beta_i ─────────────────
beta_ols = {}
for i in range(N_assets):
    sub = df3[df3['asset'] == i]
    X   = np.column_stack([np.ones(len(sub)), sub['mkt_excess'].values])
    y_i = sub['excess_ret'].values
    coef = np.linalg.lstsq(X, y_i, rcond=None)[0]
    beta_ols[i] = coef[1]   # 斜率 = beta_i

beta_ols_arr = np.array([beta_ols[i] for i in range(N_assets)])
print(f'逐资产 OLS beta 均值：{beta_ols_arr.mean():.4f}，范围：[{beta_ols_arr.min():.3f}, {beta_ols_arr.max():.3f}]')

In [ ]:
# ── Step 3：Fama-MacBeth 横截面回归 ──────────────────────
mean_ret = asset_stats['mean'].values
X_fm = np.column_stack([np.ones(N_assets), beta_ols_arr])
fm_coef = np.linalg.lstsq(X_fm, mean_ret, rcond=None)[0]
lambda_fm = fm_coef[1]   # 风险溢价
alpha_fm  = fm_coef[0]   # 截距

print(f'Fama-MacBeth 估计：')
print(f'  风险溢价 λ = {lambda_fm:.5f}')
print(f'  截距 α  = {alpha_fm:.5f}')

In [ ]:
# ── Step 4：GMM 多方程估计（利用截面误差协方差）─────────
# 矩条件：E[(R_it - R_ft) - lambda * beta_i] = 0 对每个资产 i
# 简化：直接用 WLS，权重为截面误差协方差的逆

# 计算各资产残差（使用第一步 FM 估计量）
resid_matrix = np.zeros((T_periods, N_assets))
for i in range(N_assets):
    sub = df3[df3['asset'] == i].sort_values('time')
    predicted = alpha_fm + beta_ols[i] * sub['mkt_excess'].values
    resid_matrix[:, i] = sub['excess_ret'].values - predicted

# 截面误差协方差矩阵
Sigma_e = resid_matrix.T @ resid_matrix / T_periods
try:
    Sigma_inv = np.linalg.inv(Sigma_e)
except np.linalg.LinAlgError:
    Sigma_inv = np.linalg.pinv(Sigma_e)

# GMM 估计（利用 Sigma 逆加权的横截面回归）
X_gmm = np.column_stack([np.ones(N_assets), beta_ols_arr])
A_gmm = X_gmm.T @ Sigma_inv @ X_gmm
b_gmm = X_gmm.T @ Sigma_inv @ mean_ret
gmm_coef = np.linalg.solve(A_gmm, b_gmm)
lambda_gmm = gmm_coef[1]
alpha_gmm  = gmm_coef[0]

# 标准误
n_obs = N_assets
Var_gmm = np.linalg.inv(A_gmm) / T_periods
se_lambda_gmm = np.sqrt(Var_gmm[1, 1])

# FM 标准误（简单 OLS SE）
resid_fm = mean_ret - X_fm @ fm_coef
s2_fm = resid_fm @ resid_fm / (N_assets - 2)
se_lambda_fm = np.sqrt(s2_fm * np.linalg.inv(X_fm.T @ X_fm)[1, 1])

print('=' * 55)
print(f'  方法            风险溢价 λ     标准误')
print('-' * 55)
print(f'  Fama-MacBeth    {lambda_fm:.5f}       {se_lambda_fm:.5f}')
print(f'  GMM 多方程      {lambda_gmm:.5f}       {se_lambda_gmm:.5f}')
print('=' * 55)

In [ ]:
# ── Step 5：J 检验（所有定价误差联合为零）────────────────
pricing_errors = mean_ret - gmm_coef[0] - lambda_gmm * beta_ols_arr
J3 = T_periods * pricing_errors @ Sigma_inv @ pricing_errors
df_J3 = N_assets - 2   # N*q - k = 25 - 2 = 23
p_J3  = 1 - stats.chi2.cdf(J3, df=df_J3)
print(f'J 检验（定价误差联合为零）：J = {J3:.4f}, df = {df_J3}, p = {p_J3:.4f}')
if p_J3 > 0.05:
    print('  → 不拒绝：CAPM 定价误差联合为零（模型近似成立）')
else:
    print('  → 拒绝：存在系统性定价误差')

In [ ]:
# ── 可视化对比 ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
fig.patch.set_facecolor(COLOR_BG)

ax = axes[0]
ax.set_facecolor(COLOR_BG)
ax.bar(['Fama-MacBeth', 'GMM 多方程'],
       [lambda_fm, lambda_gmm],
       yerr=[se_lambda_fm, se_lambda_gmm],
       color=[COLOR_SECONDARY, COLOR_PRIMARY], alpha=0.85,
       edgecolor='white', linewidth=1.5, width=0.4,
       capsize=6)
ax.axhline(0.006, color=COLOR_GREEN, lw=1.5, ls='--', label='真实 $\\lambda=0.006$')
ax.set_ylabel('风险溢价估计 $\\hat{\\lambda}$', fontsize=11)
ax.set_title('风险溢价估计对比', fontsize=11, fontweight='bold', color=COLOR_PRIMARY)
ax.legend(fontsize=9)
ax.spines[['top','right']].set_visible(False)

ax = axes[1]
ax.set_facecolor(COLOR_BG)
ax.scatter(beta_ols_arr, mean_ret, color=COLOR_NEUTRAL, s=50, alpha=0.7, label='资产')
beta_line = np.linspace(beta_ols_arr.min(), beta_ols_arr.max(), 100)
ax.plot(beta_line, alpha_fm + lambda_fm * beta_line,
        color=COLOR_SECONDARY, lw=2, label='Fama-MacBeth SML')
ax.plot(beta_line, alpha_gmm + lambda_gmm * beta_line,
        color=COLOR_PRIMARY, lw=2, ls='--', label='GMM SML')
ax.set_xlabel('估计 $\\hat{\\beta}_i$', fontsize=11)
ax.set_ylabel('平均超额收益率', fontsize=11)
ax.set_title('证券市场线（SML）拟合', fontsize=11, fontweight='bold', color=COLOR_PRIMARY)
ax.legend(fontsize=8)
ax.spines[['top','right']].set_visible(False)

plt.suptitle('Case 3：Fama-MacBeth vs. GMM 多方程估计', fontsize=11,
             color=COLOR_NEUTRAL, y=1.01)
plt.tight_layout()
plt.show()

print('\n结果解读：')
print(f'  • Fama-MacBeth 忽略截面误差相关性，GMM 明确利用 Σ 矩阵')
print(f'  • GMM 标准误（{se_lambda_gmm:.5f}）通常 < FM 标准误（{se_lambda_fm:.5f}）')
print(f'  • J 检验（p={p_J3:.3f}）：CAPM 定价误差联合为零，模型近似成立')

::: {.callout-note collapse="true"}
### Stata 对应代码

```stata
import delimited "./data/method_GMM_data03_asset_pricing.csv", clear

* Fama-MacBeth 回归（xtfmb 命令，需安装）
* ssc install xtfmb
xtset asset time
xtfmb excess_ret mkt_excess

* GMM 多方程估计（思路：构造 SUR 形式后用 gmm 命令）
* 完整代码见配套讲义
```
:::

---
## 综合小结

| 案例 | 方法 | 矩条件来源 | 关键发现 |
|:-----|:----|:---------|:--------|
| Case 1 | 有效 GMM vs. 2SLS | 工具变量正交性 | 异方差下 GMM 标准误更可靠，Sargan vs. Hansen J 结论可能不同 |
| Case 2a/2b | 非线性 GMM | Euler 方程 (FOC) | 不需要分布假设，直接用经济理论矩条件估计 $\beta, \gamma$ |
| Case 3 | 多方程 GMM | 多资产正交性 | 利用方程间误差相关性，比 Fama-MacBeth 更有效 |

三个案例共同印证了本章核心主线：

> **GMM 的核心思想是——用「经济理论或统计假设告诉我们应当成立的矩条件」约束参数估计，找出使样本矩条件偏离最小的那组参数值。**

无论是线性还是非线性、单方程还是多方程，GMM 提供了一个统一的估计框架；而权重矩阵 $W = \hat{S}^{-1}$ 的选择，决定了这个框架的效率上限。